In [ ]:
from worker.batch import get_portfolio_positions
from worker.database import connect
import datetime

import polars as pl
from scipy import stats

import math
import time

In [ ]:
ref_date = datetime.date(2025, 8, 25)
first_calc_date = datetime.date(2024, 9, 1)
first_price_date = datetime.date(2016, 1, 1)

s_price_dates = pl.date_range(first_price_date, ref_date, eager=True)


def _single_loop(df_positions: pl.DataFrame, df_market_data: pl.DataFrame, ptf_id: str, calc_date: datetime.date):
    df_ = (df_positions.filter(pl.col("portfolio_id").eq(ptf_id)).filter(pl.col("date").eq(calc_date))).drop(
        "portfolio_id", "date"
    )

    df_prices = df_.select("instrument_id").join(df_market_data, how="right", on=["instrument_id"])

    df_ = (
        df_
        .join(df_prices, on=["instrument_id"])
        .filter(pl.col("date") <= calc_date)
        .with_columns(value=pl.col("quantity") * pl.col("price"))
        .group_by("date")
        .agg(value=pl.col("value").sum())
        .sort("date")
        .with_columns(returns=pl.col("value").pct_change())
    )

    rets = df_["returns"]
    value = df_["value"][-1]

    assert len(rets) > 60

    vol = rets.tail(60).std()

    assert isinstance(vol, float)

    annualized_vol = vol * math.sqrt(250)
    longer_vol = rets.tail(250).std()

    hist_var = rets.tail(500).quantile(1 - 0.99)
    assert hist_var is not None

    hist_var = - hist_var

    ewma_backcast_window = 60
    ewma_backcast_vol = rets[:ewma_backcast_window].std()
    curr = ewma_backcast_vol
    ewma_vols = [curr]

    for ret in rets[ewma_backcast_window:]:
        assert isinstance(curr, float)
        curr = math.sqrt(0.94 * curr **2 + 0.06 * ret**2)
        ewma_vols.append(curr)

    ewma_vol = ewma_vols[-1]
    assert isinstance(ewma_vol, float)
    assert isinstance(longer_vol, float)

    ewma_var = float(-ewma_vol * stats.norm.ppf(1 - 0.99))
    param_var = float(-longer_vol * stats.norm.ppf(1 - 0.99))

    # Put it all together
    return pl.DataFrame(
        {
            "portfolio_id": ptf_id,
            "calc_date": calc_date,
            "value": value,
            "hist_var": hist_var,
            "ewma_var": ewma_var,
            "param_var": param_var,
            "ex_ante_vol": annualized_vol,
        }
    )    

with connect() as session:
    df_market_data = (
        pl.read_database(
            """
            SELECT
                date,
                instrument_id::TEXT as instrument_id,
                value AS price
            FROM market_data
            WHERE date <= :date
            AND data_type = 'adj_close'""",
            session,
            execute_options={"params": {"date": ref_date}},
        )
        .group_by("instrument_id")
        .map_groups(lambda df: (df.sort("date").with_columns(price=pl.col("price").forward_fill())))
    )

    df_positions = get_portfolio_positions(session, ref_date).filter(pl.col("date") >= first_calc_date)

    combinations = df_positions.select("portfolio_id", "date").unique()

    dfs = []
    for i, (ptf_id, calc_date) in enumerate(combinations.rows()):
        print(f"{i} / {len(combinations)}")
        dfs.append(_single_loop(df_positions, df_market_data, ptf_id, calc_date))


0 / 12800
1 / 12800
2 / 12800
3 / 12800
4 / 12800
5 / 12800
6 / 12800
7 / 12800
8 / 12800
9 / 12800
10 / 12800
11 / 12800
12 / 12800
13 / 12800
14 / 12800
15 / 12800
16 / 12800
17 / 12800
18 / 12800
19 / 12800
20 / 12800
21 / 12800
22 / 12800
23 / 12800
24 / 12800
25 / 12800
26 / 12800
27 / 12800
28 / 12800
29 / 12800
30 / 12800
31 / 12800
32 / 12800
33 / 12800
34 / 12800
35 / 12800
36 / 12800
37 / 12800
38 / 12800
39 / 12800
40 / 12800
41 / 12800
42 / 12800
43 / 12800
44 / 12800
45 / 12800
46 / 12800
47 / 12800
48 / 12800
49 / 12800
50 / 12800
51 / 12800
52 / 12800
53 / 12800
54 / 12800
55 / 12800
56 / 12800
57 / 12800
58 / 12800
59 / 12800
60 / 12800
61 / 12800
62 / 12800
63 / 12800
64 / 12800
65 / 12800
66 / 12800
67 / 12800
68 / 12800
69 / 12800
70 / 12800
71 / 12800
72 / 12800
73 / 12800
74 / 12800
75 / 12800
76 / 12800
77 / 12800
78 / 12800
79 / 12800
80 / 12800
81 / 12800
82 / 12800
83 / 12800
84 / 12800
85 / 12800
86 / 12800
87 / 12800
88 / 12800
89 / 12800
90 / 12800
91 / 1280

In [27]:
pl.concat(dfs)

portfolio_id,calc_date,value,hist_var,ewma_var,param_var,ex_ante_vol
str,date,"decimal[*,20]",f64,f64,f64,f64
"""b872c12a-2db2-4898-a52d-4f09e8…",2023-03-08,59330.00000000000000000000,0.031007,0.022029,0.032494,0.149976
"""ea975ea7-5bc4-4584-ad79-ea24d1…",2023-12-21,22870.00000000000000000000,0.052724,0.042521,0.040613,0.313526
"""6f7e34d1-0109-4f9e-8aa0-5d4e5f…",2024-03-29,24070.00000000000000000000,0.042105,0.033685,0.040698,0.292724
"""73b9f260-1100-469b-99a2-3a5f9e…",2023-11-13,77550.00000000000000000000,0.03698,0.033709,0.028821,0.207321
"""e01b0257-f0ea-40a3-b91d-434677…",2024-10-09,51510.00000000000000000000,0.005891,0.004846,0.007107,0.034416
…,…,…,…,…,…,…
"""0b209d5c-12ed-48af-a597-b1e22f…",2023-12-28,40230.00000000000000000000,0.03066,0.021948,0.022869,0.150429
"""a2d06bf1-3dd4-4293-87a6-62cd3a…",2022-11-07,42430.00000000000000000000,0.028162,0.029435,0.028635,0.203926
"""74623cfa-69a7-43c4-a6e5-3beb9b…",2023-03-23,51730.00000000000000000000,0.04047,0.034903,0.040774,0.204138
